# Legal Document RAG Ingestion Pipeline V4
## Kenya Constitution & Acts - Production-Ready Implementation

---

### 📋 Overview

This notebook implements a **production-ready, step-by-step pipeline** for ingesting legal documents (Kenyan corpus), extracting structure & metadata, creating hierarchy-aware chunks, generating embeddings, and upserting into `pgvector`.

### 🗄️ Database: `legalRAG_V4_db`

### 📁 Source Documents: `rag-dataset/legal documents/`

---

### ⚙️ Requirements

**ALWAYS activate the conda environment first:**

```bash
conda activate LLMTuning311
```

**Dependencies:**
- PostgreSQL with pgvector extension
- Local OCR: **EasyOCR** (GPU-accelerated with MPS/CUDA support)
- Local embedding model: nomic-embed-text (via Ollama)
- Python packages: see requirements below

**Install required packages:**
```bash
# Core dependencies
pip install easyocr psycopg2-binary pgvector PyMuPDF pdfplumber Pillow opencv-python numpy pandas langchain-ollama langchain-community tiktoken scikit-learn tqdm python-dotenv

# PyTorch with MPS support (for Apple Silicon)
# This is usually pre-installed in your conda environment
pip install torch torchvision
```

**GPU Support:**
- **Apple Silicon (M1/M2/M3)**: EasyOCR automatically uses MPS (Metal Performance Shaders) 🚀
- **NVIDIA GPU**: EasyOCR automatically uses CUDA if available 🚀
- **CPU fallback**: Works on all systems

---

### 🔄 Pipeline Flow

1. Setup: imports, env, device detection
2. Configuration & Constants
3. Ingestion & File Hashing
4. Per-page classification: digital vs scanned vs hybrid
5. Layout parsing & OCR (GPU-accelerated: MPS/CUDA/CPU)
6. Noise removal & normalization
7. Structure detection: sections, articles, tables
8. Metadata extraction & canonical fields
9. Chunking strategy (hierarchy-aware)
10. Embedding generation & pgvector upsert
11. Versioning & deduplication
12. QA, logging, monitoring
13. Retriever config / System prompt / RAG usage

---

### ⚠️ Important Rules

- **Stop on failure:** Pipeline stops on any error and logs to DB
- **No automatic progression:** Wait for manual approval after each major step
- **Local-first:** Uses only locally hosted models (no cloud services)
- **Device-aware:** Auto-detects CUDA → MPS → CPU, OCR fully GPU-accelerated
- **No silent fallbacks:** Missing resources = error + stop

---

## 1 - Setup: imports, env, device detection

In [31]:
# Activate conda environment first!
# Run in terminal: conda activate LLMTuning311

import os
import sys
import warnings
import hashlib
import json
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from datetime import datetime
import uuid

# Database
import psycopg2
from psycopg2.extras import Json, RealDictCursor
from pgvector.psycopg2 import register_vector

# PDF processing
import fitz  # PyMuPDF
import pdfplumber

# OCR - EasyOCR (MPS/CUDA/CPU-accelerated, full device awareness)
import easyocr
from PIL import Image
import cv2

# Data processing
import numpy as np
import pandas as pd
import re
from collections import defaultdict

# ML & Embeddings
try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("⚠️  PyTorch not available - CPU-only mode")

from langchain_ollama import OllamaEmbeddings
import tiktoken
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# Environment
from dotenv import load_dotenv
load_dotenv()

warnings.filterwarnings('ignore')

# Device detection
def detect_device() -> Tuple[str, bool]:
    """
    Detect and return best available device: CUDA → MPS → CPU
    Returns: (device_name, use_gpu_for_ocr)
    """
    if not TORCH_AVAILABLE:
        print("✅ Using CPU (PyTorch not available)")
        return "cpu", False
    
    if torch.cuda.is_available():
        device = "cuda"
        gpu_name = torch.cuda.get_device_name(0)
        print(f"✅ CUDA GPU detected: {gpu_name}")
        return device, True
    elif torch.backends.mps.is_available():
        device = "mps"
        print("✅ Apple Silicon MPS detected")
        return device, True  # EasyOCR supports MPS!
    else:
        device = "cpu"
        print("✅ Using CPU")
        return device, False

DEVICE, USE_GPU = detect_device()

# Database configuration (PostgreSQL converts to lowercase)
DB_NAME = 'legalrag_v4_db'
DB_USER = os.getenv('DB_USER', os.getenv('USER'))
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_PASSWORD = os.getenv('DB_PASSWORD', '')

print(f"\n📊 Database Configuration:")
print(f"   Database: {DB_NAME}")
print(f"   Host: {DB_HOST}:{DB_PORT}")
print(f"   User: {DB_USER}")

# Initialize EasyOCR with device-aware GPU acceleration
try:
    print(f"\n🔧 Initializing EasyOCR...")
    print(f"   Target device: {DEVICE}")
    print(f"   GPU acceleration: {'Enabled ✅' if USE_GPU else 'Disabled (CPU mode)'}")
    
    # EasyOCR automatically uses the best available device
    # gpu=True enables CUDA or MPS depending on what's available
    OCR_ENGINE = easyocr.Reader(
        ['en'],              # English language
        gpu=USE_GPU,         # Enable GPU (CUDA or MPS)
        verbose=False        # Suppress verbose logs
    )
    
    print(f"✅ EasyOCR initialized successfully")
    print(f"   → Active device: {DEVICE}")
    if USE_GPU:
        if DEVICE == 'cuda':
            print(f"   → Using NVIDIA CUDA acceleration 🚀")
        elif DEVICE == 'mps':
            print(f"   → Using Apple Metal (MPS) acceleration 🚀")
    else:
        print(f"   → Using CPU (no GPU detected)")
    
except Exception as e:
    print(f"\n❌ EasyOCR initialization failed: {e}")
    print("   Install with: pip install easyocr torch torchvision")
    raise

print(f"\n✅ Environment validated: device={DEVICE}, gpu_enabled={USE_GPU}")


[2025-11-20 14:01:31,673] [ WARNING] easyocr.py:251 - Downloading detection model, please wait. This may take several minutes depending upon your network connection.


✅ Apple Silicon MPS detected

📊 Database Configuration:
   Database: legalrag_v4_db
   Host: localhost:5432
   User: quest

🔧 Initializing EasyOCR...
   Target device: mps
   GPU acceleration: Enabled ✅


[2025-11-20 14:01:58,894] [ WARNING] easyocr.py:176 - Downloading recognition model, please wait. This may take several minutes depending upon your network connection.


✅ EasyOCR initialized successfully
   → Active device: mps
   → Using Apple Metal (MPS) acceleration 🚀

✅ Environment validated: device=mps, gpu_enabled=True


In [41]:
# Database connection and table creation

def get_db_connection(register_vec=False):
    """Create database connection with optional pgvector support"""
    try:
        conn = psycopg2.connect(
            dbname=DB_NAME,
            user=DB_USER,
            password=DB_PASSWORD,
            host=DB_HOST,
            port=DB_PORT
        )
        if register_vec:
            register_vector(conn)
        return conn
    except psycopg2.OperationalError as e:
        if "does not exist" in str(e):
            print(f"❌ Database '{DB_NAME}' does not exist.")
            print(f"\n   Creating database '{DB_NAME}'...")
            
            # Connect to default postgres database to create new one
            conn = psycopg2.connect(
                dbname='postgres',
                user=DB_USER,
                password=DB_PASSWORD,
                host=DB_HOST,
                port=DB_PORT
            )
            conn.autocommit = True
            with conn.cursor() as cur:
                try:
                    cur.execute(f"CREATE DATABASE {DB_NAME}")
                    print(f"✅ Database '{DB_NAME}' created")
                except psycopg2.errors.DuplicateDatabase:
                    print(f"✅ Database '{DB_NAME}' already exists")
            conn.close()
            
            # Reconnect to new database
            conn = psycopg2.connect(
                dbname=DB_NAME,
                user=DB_USER,
                password=DB_PASSWORD,
                host=DB_HOST,
                port=DB_PORT
            )
            return conn
        else:
            raise

def create_tables(conn):
    """Create all required tables for the pipeline"""
    with conn.cursor() as cur:
        # Enable pgvector extension
        cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
        conn.commit()
    
    # Now register vector type after extension is created
    register_vector(conn)
    
    with conn.cursor() as cur:
        # documents table
        cur.execute("""
            CREATE TABLE IF NOT EXISTS documents (
                doc_id TEXT PRIMARY KEY,
                short_title TEXT,
                citation TEXT,
                jurisdiction TEXT DEFAULT 'Kenya',
                doc_type TEXT,
                created_at TIMESTAMP DEFAULT now()
            )
        """)
        
        # document_versions table
        cur.execute("""
            CREATE TABLE IF NOT EXISTS document_versions (
                version_id TEXT PRIMARY KEY,
                doc_id TEXT REFERENCES documents(doc_id),
                file_hash TEXT UNIQUE,
                text_hash TEXT,
                shingle_hash TEXT,
                document_date DATE,
                version_info TEXT,
                status TEXT CHECK (status IN ('current','superseded','draft')) DEFAULT 'draft',
                superseded_by TEXT,
                ocr_conf_doc REAL,
                page_count INT,
                is_scanned BOOLEAN,
                has_tables BOOLEAN,
                has_amendments BOOLEAN,
                file_path TEXT,
                file_size BIGINT,
                mime_type TEXT,
                sections JSONB,
                metadata JSONB,
                created_at TIMESTAMP DEFAULT now()
            )
        """)
        
        # embeddings table with pgvector
        cur.execute("""
            CREATE TABLE IF NOT EXISTS embeddings (
                chunk_id TEXT PRIMARY KEY,
                version_id TEXT REFERENCES document_versions(version_id),
                chunk_text TEXT NOT NULL,
                embedding_vector vector(768),
                chunk_type TEXT,
                section_ref TEXT,
                extra_metadata JSONB,
                created_at TIMESTAMP DEFAULT now()
            )
        """)
        
        # ingest_errors table
        cur.execute("""
            CREATE TABLE IF NOT EXISTS ingest_errors (
                id SERIAL PRIMARY KEY,
                version_id TEXT,
                step TEXT,
                error_message TEXT,
                error_payload JSONB,
                occurred_at TIMESTAMP DEFAULT now()
            )
        """)
        
        # pipeline_logs table
        cur.execute("""
            CREATE TABLE IF NOT EXISTS pipeline_logs (
                id SERIAL PRIMARY KEY,
                version_id TEXT,
                event TEXT,
                payload JSONB,
                logged_at TIMESTAMP DEFAULT now()
            )
        """)
        
        # Create index on embeddings
        cur.execute("""
            CREATE INDEX IF NOT EXISTS embeddings_vector_idx 
            ON embeddings USING ivfflat (embedding_vector vector_cosine_ops)
            WITH (lists = 100)
        """)
        
        conn.commit()
    
    print("✅ All tables created successfully")

# Initialize database
try:
    conn = get_db_connection(register_vec=False)
    print(f"✅ Connected to PostgreSQL database: {DB_NAME}")
    
    create_tables(conn)
    
    # Log initialization
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO pipeline_logs (event, payload)
            VALUES (%s, %s)
        """, ('pipeline_init', Json({'device': DEVICE, 'timestamp': datetime.now().isoformat()})))
        conn.commit()
    
    print("\n✅ Setup complete - ready for ingestion")
    
except Exception as e:
    print(f"\n❌ Setup failed: {e}")
    raise


✅ Connected to PostgreSQL database: legalrag_v4_db
✅ All tables created successfully

✅ Setup complete - ready for ingestion


---
### ✋ CHECKPOINT 1

**Setup complete!**

- ✅ Device detected and configured
- ✅ Database `legalRAG_V4_db` created/connected
- ✅ All tables initialized
- ✅ Tesseract OCR verified

**⚠️ Do NOT proceed to the next cell until instructed.**

---

## 2 - Configuration & Constants

In [33]:
# Pipeline configuration

class PipelineConfig:
    """Central configuration for the RAG ingestion pipeline"""
    
    # Paths
    SOURCE_DIR = Path("rag-dataset/legal documents")
    STORAGE_DIR = Path("pipeline_storage")
    RAW_FILES_DIR = STORAGE_DIR / "raw_files"
    OCR_OUTPUT_DIR = STORAGE_DIR / "ocr_output"
    NORMALIZED_DIR = STORAGE_DIR / "normalized"
    
    # EasyOCR configuration
    OCR_ENGINE_NAME = 'EasyOCR'
    OCR_LANG = ['en']  # Language list
    OCR_USE_GPU = USE_GPU  # Device-aware GPU setting (CUDA/MPS/CPU)
    OCR_DEVICE = DEVICE  # Current device (cuda/mps/cpu)
    OCR_CONFIDENCE_THRESHOLD = 0.6  # Minimum acceptable OCR confidence (0-1)
    OCR_BATCH_SIZE = 1  # Batch size for OCR processing
    
    # Embedding configuration
    EMBEDDING_MODEL = 'nomic-embed-text'
    EMBEDDING_BASE_URL = 'http://localhost:11434'
    EMBEDDING_DIM = 768
    
    # Chunking configuration
    MIN_CHUNK_TOKENS = 400
    MAX_CHUNK_TOKENS = 800
    CHUNK_OVERLAP_PERCENTAGE = 0.15
    
    # Document classification thresholds
    DIGITAL_TEXT_THRESHOLD = 50  # Min chars to consider page as digital
    SCANNED_CONFIDENCE_THRESHOLD = 0.5  # Threshold for scanned classification
    
    # Jurisdiction and document types
    DEFAULT_JURISDICTION = 'Kenya'
    ALLOWED_DOC_TYPES = ['Constitution', 'Act', 'Regulation', 'Notice', 'Corrigendum', 'Guideline']
    
    # Ingestion mode
    BATCH_SIZE = 5  # For batch processing
    MODE = 'batch'  # or 'streaming'
    
    # Versioning
    SHINGLE_SIZE = 5  # For near-duplicate detection
    SIMILARITY_THRESHOLD = 0.95  # Threshold for marking as duplicate

# Create directories
for dir_path in [PipelineConfig.STORAGE_DIR, PipelineConfig.RAW_FILES_DIR, 
                 PipelineConfig.OCR_OUTPUT_DIR, PipelineConfig.NORMALIZED_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print("📋 Pipeline Configuration:")
print(f"   Source: {PipelineConfig.SOURCE_DIR}")
print(f"   Storage: {PipelineConfig.STORAGE_DIR}")
print(f"   OCR Engine: {PipelineConfig.OCR_ENGINE_NAME}")
print(f"   OCR Languages: {', '.join(PipelineConfig.OCR_LANG)}")
print(f"   OCR Device: {PipelineConfig.OCR_DEVICE}")
print(f"   OCR GPU Acceleration: {'Enabled ✅' if PipelineConfig.OCR_USE_GPU else 'Disabled (CPU)'}")
if PipelineConfig.OCR_USE_GPU:
    accel_type = "NVIDIA CUDA" if PipelineConfig.OCR_DEVICE == 'cuda' else "Apple Metal (MPS)" if PipelineConfig.OCR_DEVICE == 'mps' else "Unknown"
    print(f"   OCR Acceleration Type: {accel_type}")
print(f"   OCR Confidence Threshold: {PipelineConfig.OCR_CONFIDENCE_THRESHOLD:.0%}")
print(f"   Embedding Model: {PipelineConfig.EMBEDDING_MODEL}")
print(f"   Chunk Size: {PipelineConfig.MIN_CHUNK_TOKENS}-{PipelineConfig.MAX_CHUNK_TOKENS} tokens")
print(f"   Jurisdiction: {PipelineConfig.DEFAULT_JURISDICTION}")
print(f"   Mode: {PipelineConfig.MODE}")

# Validate Ollama is running
try:
    test_embeddings = OllamaEmbeddings(
        model=PipelineConfig.EMBEDDING_MODEL,
        base_url=PipelineConfig.EMBEDDING_BASE_URL
    )
    test_vec = test_embeddings.embed_query("test")
    print(f"\n✅ Ollama embeddings verified (dim: {len(test_vec)})")
except Exception as e:
    print(f"\n❌ Ollama connection failed: {e}")
    print("   Ensure Ollama is running: ollama serve")
    print(f"   Pull model: ollama pull {PipelineConfig.EMBEDDING_MODEL}")
    raise

# Log configuration to database
with conn.cursor() as cur:
    cur.execute("""
        INSERT INTO pipeline_logs (event, payload)
        VALUES (%s, %s)
    """, ('config_loaded', Json({
        'embedding_model': PipelineConfig.EMBEDDING_MODEL,
        'chunk_range': f'{PipelineConfig.MIN_CHUNK_TOKENS}-{PipelineConfig.MAX_CHUNK_TOKENS}',
        'ocr_engine': PipelineConfig.OCR_ENGINE_NAME,
        'ocr_device': PipelineConfig.OCR_DEVICE,
        'ocr_gpu': PipelineConfig.OCR_USE_GPU,
        'ocr_threshold': PipelineConfig.OCR_CONFIDENCE_THRESHOLD,
        'device': DEVICE
    })))
    conn.commit()

print("\n✅ Configuration validated and logged")


📋 Pipeline Configuration:
   Source: rag-dataset/legal documents
   Storage: pipeline_storage
   OCR Engine: EasyOCR
   OCR Languages: en
   OCR Device: mps
   OCR GPU Acceleration: Enabled ✅
   OCR Acceleration Type: Apple Metal (MPS)
   OCR Confidence Threshold: 60%
   Embedding Model: nomic-embed-text
   Chunk Size: 400-800 tokens
   Jurisdiction: Kenya
   Mode: batch

✅ Ollama embeddings verified (dim: 768)

✅ Configuration validated and logged

✅ Ollama embeddings verified (dim: 768)

✅ Configuration validated and logged


---
### ✋ CHECKPOINT 2

**Configuration complete!**

- ✅ Pipeline directories created
- ✅ Ollama embeddings verified
- ✅ Configuration logged to database

**⚠️ Do NOT proceed to the next cell until instructed.**

---

## 3 - Ingestion & File Hashing

In [34]:
# Helper functions for ingestion

def compute_file_hash(file_path: Path) -> str:
    """Compute SHA256 hash of a file"""
    sha256_hash = hashlib.sha256()
    with open(file_path, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

def compute_text_hash(text: str) -> str:
    """Compute SHA256 hash of text content"""
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def generate_shingles(text: str, k: int = 5) -> set:
    """Generate k-shingles (k-grams) from text for similarity detection"""
    words = text.lower().split()
    shingles = set()
    for i in range(len(words) - k + 1):
        shingle = ' '.join(words[i:i+k])
        shingles.add(shingle)
    return shingles

def compute_shingle_hash(text: str, k: int = 5) -> str:
    """Compute hash of text shingles for near-duplicate detection"""
    shingles = generate_shingles(text, k)
    # Sort shingles for consistent hashing
    shingles_str = '|'.join(sorted(shingles))
    return hashlib.sha256(shingles_str.encode('utf-8')).hexdigest()

def log_error(conn, version_id: str, step: str, error_msg: str, payload: dict = None):
    """Log error to database"""
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO ingest_errors (version_id, step, error_message, error_payload)
            VALUES (%s, %s, %s, %s)
        """, (version_id, step, error_msg, Json(payload or {})))
        conn.commit()

def log_event(conn, version_id: Optional[str], event: str, payload: dict):
    """Log event to pipeline_logs"""
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO pipeline_logs (version_id, event, payload)
            VALUES (%s, %s, %s)
        """, (version_id, event, Json(payload)))
        conn.commit()

print("✅ Helper functions loaded")

✅ Helper functions loaded


In [35]:
# Discover and list PDF files

def discover_pdf_files(source_dir: Path) -> List[Path]:
    """Discover all PDF files in source directory"""
    pdf_files = list(source_dir.glob("*.pdf"))
    return sorted(pdf_files)

# Discover PDFs
pdf_files = discover_pdf_files(PipelineConfig.SOURCE_DIR)

if not pdf_files:
    error_msg = f"No PDF files found in {PipelineConfig.SOURCE_DIR}"
    print(f"❌ {error_msg}")
    log_error(conn, None, "file_discovery", error_msg)
    raise FileNotFoundError(error_msg)

print(f"📁 Discovered {len(pdf_files)} PDF files:")
for i, pdf_path in enumerate(pdf_files, 1):
    file_size = pdf_path.stat().st_size / 1024  # KB
    print(f"   {i}. {pdf_path.name} ({file_size:.1f} KB)")

log_event(conn, None, "files_discovered", {
    'count': len(pdf_files),
    'files': [p.name for p in pdf_files]
})

print(f"\n✅ File discovery complete")

📁 Discovered 4 PDF files:
   1. Acts Published for the Implementation of the Constitution Corrigenda.pdf (26905.6 KB)
   2. Adjustment of Rates for Inflation (3).pdf (30.1 KB)
   3. Advocates Act.pdf (529.0 KB)
   4. TheConstitutionOfKenya.pdf (1273.9 KB)

✅ File discovery complete


In [36]:
# Ingest files: compute hashes and create initial database records

def ingest_file(file_path: Path, conn) -> Tuple[str, str, dict]:
    """
    Ingest a single PDF file: compute hashes, create DB records
    
    Returns:
        Tuple of (doc_id, version_id, metadata)
    """
    try:
        # Extract filename without extension as short title (will be refined later)
        filename = file_path.stem
        
        # Compute file hash
        file_hash = compute_file_hash(file_path)
        file_size = file_path.stat().st_size
        
        # Check if file already ingested (by hash)
        with conn.cursor() as cur:
            cur.execute("""
                SELECT version_id, doc_id FROM document_versions 
                WHERE file_hash = %s
            """, (file_hash,))
            existing = cur.fetchone()
            
            if existing:
                version_id, doc_id = existing
                print(f"   ⚠️  File already ingested: {filename}")
                print(f"      Existing version_id: {version_id}")
                return doc_id, version_id, {'status': 'skipped', 'reason': 'duplicate_hash'}
        
        # Generate IDs
        doc_id = f"doc_{uuid.uuid4().hex[:12]}"
        version_id = f"v_{uuid.uuid4().hex[:12]}"
        
        # Get basic PDF metadata using PyMuPDF
        doc = fitz.open(file_path)
        page_count = len(doc)
        pdf_metadata = doc.metadata
        doc.close()
        
        # Insert into documents table
        with conn.cursor() as cur:
            cur.execute("""
                INSERT INTO documents (doc_id, short_title, jurisdiction)
                VALUES (%s, %s, %s)
                ON CONFLICT (doc_id) DO NOTHING
            """, (doc_id, filename, PipelineConfig.DEFAULT_JURISDICTION))
            
            # Insert into document_versions table
            cur.execute("""
                INSERT INTO document_versions 
                (version_id, doc_id, file_hash, file_path, file_size, 
                 mime_type, page_count, status, metadata)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
            """, (
                version_id, doc_id, file_hash, str(file_path), file_size,
                'application/pdf', page_count, 'draft',
                Json({'pdf_metadata': pdf_metadata, 'original_filename': file_path.name})
            ))
            
            conn.commit()
        
        # Copy file to storage
        storage_path = PipelineConfig.RAW_FILES_DIR / f"{version_id}_{file_path.name}"
        import shutil
        shutil.copy2(file_path, storage_path)
        
        metadata = {
            'status': 'ingested',
            'page_count': page_count,
            'file_size': file_size,
            'storage_path': str(storage_path)
        }
        
        return doc_id, version_id, metadata
        
    except Exception as e:
        error_msg = f"Failed to ingest {file_path.name}: {str(e)}"
        print(f"   ❌ {error_msg}")
        log_error(conn, None, "file_ingestion", error_msg, {'file': str(file_path)})
        raise

# Ingest all discovered files
print(f"\n📥 Ingesting {len(pdf_files)} files...\n")

ingestion_results = []

for i, pdf_path in enumerate(pdf_files, 1):
    print(f"{i}/{len(pdf_files)}: {pdf_path.name}")
    
    try:
        doc_id, version_id, metadata = ingest_file(pdf_path, conn)
        
        ingestion_results.append({
            'file': pdf_path.name,
            'doc_id': doc_id,
            'version_id': version_id,
            'status': metadata.get('status'),
            'page_count': metadata.get('page_count'),
            'file_size_kb': metadata.get('file_size', 0) / 1024
        })
        
        # Log successful ingestion
        if metadata.get('status') == 'ingested':
            log_event(conn, version_id, "file_ingested", {
                'file': pdf_path.name,
                'doc_id': doc_id,
                'page_count': metadata.get('page_count'),
                'file_size': metadata.get('file_size')
            })
            print(f"   ✅ Ingested: {doc_id} / {version_id}")
        
    except Exception as e:
        print(f"   ❌ Failed: {e}")
        ingestion_results.append({
            'file': pdf_path.name,
            'status': 'failed',
            'error': str(e)
        })
        # Continue with next file
        continue
    
    print()

# Create summary DataFrame
df_results = pd.DataFrame(ingestion_results)

print(f"\n{'='*60}")
print(f"📊 Ingestion Summary")
print(f"{'='*60}")
print(f"Total files: {len(pdf_files)}")
print(f"Successfully ingested: {len(df_results[df_results['status'] == 'ingested'])}")
print(f"Skipped (duplicates): {len(df_results[df_results['status'] == 'skipped'])}")
print(f"Failed: {len(df_results[df_results['status'] == 'failed'])}")
print(f"{'='*60}\n")

# Display results table
print(df_results.to_string(index=False))

# Save results to disk
results_path = PipelineConfig.STORAGE_DIR / "ingestion_results.csv"
df_results.to_csv(results_path, index=False)
print(f"\n✅ Results saved to: {results_path}")

# Check if any files failed
if len(df_results[df_results['status'] == 'failed']) > 0:
    error_msg = "Some files failed to ingest. Check errors above."
    print(f"\n❌ {error_msg}")
    log_error(conn, None, "ingestion_complete", error_msg, 
              {'failed_files': df_results[df_results['status'] == 'failed']['file'].tolist()})
    raise RuntimeError(error_msg)

print(f"\n✅ Ingestion & File Hashing complete!")


📥 Ingesting 4 files...

1/4: Acts Published for the Implementation of the Constitution Corrigenda.pdf
   ⚠️  File already ingested: Acts Published for the Implementation of the Constitution Corrigenda
      Existing version_id: v_2b308b60b512

2/4: Adjustment of Rates for Inflation (3).pdf
   ⚠️  File already ingested: Adjustment of Rates for Inflation (3)
      Existing version_id: v_f87dfad31787

3/4: Advocates Act.pdf
   ⚠️  File already ingested: Advocates Act
      Existing version_id: v_7f22ccad09a4

4/4: TheConstitutionOfKenya.pdf
   ⚠️  File already ingested: TheConstitutionOfKenya
      Existing version_id: v_a7cab3e9dbb6


📊 Ingestion Summary
Total files: 4
Successfully ingested: 0
Skipped (duplicates): 4
Failed: 0

                                                                    file           doc_id     version_id  status page_count  file_size_kb
Acts Published for the Implementation of the Constitution Corrigenda.pdf doc_c5095730309f v_2b308b60b512 skipped       None  

---
### ✋ CHECKPOINT 3

**Ingestion & File Hashing complete!**

- ✅ PDF files discovered
- ✅ File hashes computed (SHA256)
- ✅ Initial database records created
- ✅ Files copied to storage
- ✅ Results logged

**Next step:** Per-page classification (digital vs scanned vs hybrid)

**⚠️ Do NOT proceed to the next cell until instructed.**

---

## 4 - Per-page classification: digital vs scanned vs hybrid

In [37]:
# Page classification functions

def classify_page_type(page, page_num: int) -> dict:
    """
    Classify a PDF page as digital, scanned, or hybrid
    
    Args:
        page: PyMuPDF page object
        page_num: Page number (0-indexed)
    
    Returns:
        dict with classification results
    """
    # Extract text using PyMuPDF
    text = page.get_text()
    text_length = len(text.strip())
    
    # Get page dimensions
    rect = page.rect
    width, height = rect.width, rect.height
    
    # Check for images
    image_list = page.get_images(full=True)
    has_images = len(image_list) > 0
    image_count = len(image_list)
    
    # Calculate image coverage (if images present)
    image_coverage = 0.0
    if has_images:
        total_image_area = 0
        page_area = width * height
        for img in image_list:
            # Get image bounding box
            xref = img[0]
            try:
                img_rect = page.get_image_bbox(img)
                img_area = abs(img_rect.width * img_rect.height)
                total_image_area += img_area
            except:
                pass
        if page_area > 0:
            image_coverage = (total_image_area / page_area) * 100
    
    # Classification logic
    if text_length >= PipelineConfig.DIGITAL_TEXT_THRESHOLD:
        if image_coverage > 80:
            page_type = "hybrid"  # Has text layer but mostly images
        else:
            page_type = "digital"  # PDF with extractable text
    else:
        if has_images or image_coverage > 50:
            page_type = "scanned"  # Image-based PDF requiring OCR
        else:
            page_type = "digital"  # Might be empty page or minimal text
    
    return {
        'page_num': page_num,
        'page_type': page_type,
        'text_length': text_length,
        'has_images': has_images,
        'image_count': image_count,
        'image_coverage': round(image_coverage, 2),
        'needs_ocr': page_type in ['scanned', 'hybrid']
    }

def classify_document_pages(file_path: Path, version_id: str, conn) -> dict:
    """
    Classify all pages in a document
    
    Returns:
        dict with classification summary
    """
    try:
        doc = fitz.open(file_path)
        
        page_classifications = []
        
        for page_num in range(len(doc)):
            page = doc[page_num]
            classification = classify_page_type(page, page_num)
            page_classifications.append(classification)
        
        doc.close()
        
        # Aggregate statistics
        total_pages = len(page_classifications)
        digital_pages = sum(1 for p in page_classifications if p['page_type'] == 'digital')
        scanned_pages = sum(1 for p in page_classifications if p['page_type'] == 'scanned')
        hybrid_pages = sum(1 for p in page_classifications if p['page_type'] == 'hybrid')
        needs_ocr_count = sum(1 for p in page_classifications if p['needs_ocr'])
        
        # Determine document-level classification
        if scanned_pages == total_pages:
            doc_classification = "fully_scanned"
        elif digital_pages == total_pages:
            doc_classification = "fully_digital"
        else:
            doc_classification = "hybrid"
        
        is_scanned = doc_classification in ["fully_scanned", "hybrid"]
        
        summary = {
            'version_id': version_id,
            'total_pages': total_pages,
            'digital_pages': digital_pages,
            'scanned_pages': scanned_pages,
            'hybrid_pages': hybrid_pages,
            'needs_ocr_count': needs_ocr_count,
            'doc_classification': doc_classification,
            'is_scanned': is_scanned,
            'page_classifications': page_classifications
        }
        
        # Update database with classification results
        with conn.cursor() as cur:
            cur.execute("""
                UPDATE document_versions
                SET is_scanned = %s,
                    metadata = COALESCE(metadata, '{}'::jsonb) || %s::jsonb
                WHERE version_id = %s
            """, (
                is_scanned,
                json.dumps({
                    'page_classifications': page_classifications,
                    'doc_classification': doc_classification,
                    'classification_summary': {
                        'digital_pages': digital_pages,
                        'scanned_pages': scanned_pages,
                        'hybrid_pages': hybrid_pages,
                        'needs_ocr_count': needs_ocr_count
                    }
                }),
                version_id
            ))
            conn.commit()
        
        return summary
        
    except Exception as e:
        error_msg = f"Page classification failed: {str(e)}"
        log_error(conn, version_id, "page_classification", error_msg, {'file': str(file_path)})
        raise

print("✅ Page classification functions loaded")

✅ Page classification functions loaded


In [38]:
# Classify all ingested documents

# Get list of documents to classify
with conn.cursor(cursor_factory=RealDictCursor) as cur:
    cur.execute("""
        SELECT version_id, file_path, doc_id
        FROM document_versions
        WHERE status = 'draft'
        ORDER BY version_id
    """)
    documents_to_classify = cur.fetchall()

if not documents_to_classify:
    error_msg = "No documents found in 'draft' status for classification"
    print(f"❌ {error_msg}")
    log_error(conn, None, "page_classification", error_msg)
    raise RuntimeError(error_msg)

print(f"📄 Classifying {len(documents_to_classify)} documents...\n")

classification_results = []

for doc in tqdm(documents_to_classify, desc="Classifying pages"):
    version_id = doc['version_id']
    file_path = Path(doc['file_path'])
    
    try:
        # Check if file exists in storage
        storage_path = PipelineConfig.RAW_FILES_DIR / f"{version_id}_{file_path.name}"
        if not storage_path.exists():
            # Try original path
            storage_path = file_path
        
        if not storage_path.exists():
            raise FileNotFoundError(f"File not found: {storage_path}")
        
        print(f"\n📖 {file_path.name}")
        summary = classify_document_pages(storage_path, version_id, conn)
        
        classification_results.append({
            'file': file_path.name,
            'version_id': version_id,
            'total_pages': summary['total_pages'],
            'digital': summary['digital_pages'],
            'scanned': summary['scanned_pages'],
            'hybrid': summary['hybrid_pages'],
            'needs_ocr': summary['needs_ocr_count'],
            'classification': summary['doc_classification'],
            'is_scanned': summary['is_scanned']
        })
        
        # Log event
        log_event(conn, version_id, "pages_classified", {
            'total_pages': summary['total_pages'],
            'doc_classification': summary['doc_classification'],
            'needs_ocr_count': summary['needs_ocr_count']
        })
        
        print(f"   Classification: {summary['doc_classification']}")
        print(f"   Digital pages: {summary['digital_pages']}")
        print(f"   Scanned pages: {summary['scanned_pages']}")
        print(f"   Hybrid pages: {summary['hybrid_pages']}")
        print(f"   Needs OCR: {summary['needs_ocr_count']}")
        
    except Exception as e:
        print(f"   ❌ Classification failed: {e}")
        classification_results.append({
            'file': file_path.name,
            'version_id': version_id,
            'status': 'failed',
            'error': str(e)
        })
        continue

# Create summary DataFrame
df_classification = pd.DataFrame(classification_results)

print(f"\n{'='*70}")
print(f"📊 Page Classification Summary")
print(f"{'='*70}")
print(f"Total documents: {len(documents_to_classify)}")
print(f"Total pages: {df_classification['total_pages'].sum()}")
print(f"Digital pages: {df_classification['digital'].sum()}")
print(f"Scanned pages: {df_classification['scanned'].sum()}")
print(f"Hybrid pages: {df_classification['hybrid'].sum()}")
print(f"Pages needing OCR: {df_classification['needs_ocr'].sum()}")
print(f"{'='*70}\n")

# Display detailed results
print(df_classification.to_string(index=False))

# Save results
classification_path = PipelineConfig.STORAGE_DIR / "classification_results.csv"
df_classification.to_csv(classification_path, index=False)
print(f"\n✅ Classification results saved to: {classification_path}")

# Check for failures
if 'status' in df_classification.columns:
    failed = df_classification[df_classification['status'] == 'failed']
    if len(failed) > 0:
        error_msg = f"{len(failed)} document(s) failed classification"
        print(f"\n❌ {error_msg}")
        log_error(conn, None, "classification_complete", error_msg, 
                  {'failed': failed['file'].tolist()})
        raise RuntimeError(error_msg)

print(f"\n✅ Page classification complete!")

📄 Classifying 4 documents...



Classifying pages:   0%|          | 0/4 [00:00<?, ?it/s]


📖 Acts Published for the Implementation of the Constitution Corrigenda.pdf


Classifying pages:  25%|██▌       | 1/4 [00:00<00:01,  2.01it/s]

   Classification: hybrid
   Digital pages: 0
   Scanned pages: 0
   Hybrid pages: 80
   Needs OCR: 80

📖 Advocates Act.pdf


Classifying pages:  50%|█████     | 2/4 [00:00<00:00,  3.01it/s]

   Classification: fully_digital
   Digital pages: 38
   Scanned pages: 0
   Hybrid pages: 0
   Needs OCR: 0

📖 TheConstitutionOfKenya.pdf


Classifying pages: 100%|██████████| 4/4 [00:02<00:00,  1.84it/s]

   Classification: hybrid
   Digital pages: 163
   Scanned pages: 3
   Hybrid pages: 0
   Needs OCR: 3

📖 Adjustment of Rates for Inflation (3).pdf
   Classification: hybrid
   Digital pages: 0
   Scanned pages: 0
   Hybrid pages: 2
   Needs OCR: 2

📊 Page Classification Summary
Total documents: 4
Total pages: 286
Digital pages: 201
Scanned pages: 3
Hybrid pages: 82
Pages needing OCR: 85

                                                                    file     version_id  total_pages  digital  scanned  hybrid  needs_ocr classification  is_scanned
Acts Published for the Implementation of the Constitution Corrigenda.pdf v_2b308b60b512           80        0        0      80         80         hybrid        True
                                                       Advocates Act.pdf v_7f22ccad09a4           38       38        0       0          0  fully_digital       False
                                              TheConstitutionOfKenya.pdf v_a7cab3e9dbb6          166      163    

---
### ✋ CHECKPOINT 4

**Per-page classification complete!**

- ✅ All pages analyzed for text/image content
- ✅ Pages classified as: digital, scanned, or hybrid
- ✅ OCR requirements identified
- ✅ Classification metadata stored in database
- ✅ Results saved to CSV

**Next step:** Layout parsing & OCR (local only)

**⚠️ Do NOT proceed to the next cell until instructed.**

---

## 5 - Layout parsing & OCR (local only)

In [42]:
# OCR and layout parsing functions

def extract_text_digital(page) -> Tuple[str, dict]:
    """Extract text from digital PDF page"""
    text = page.get_text()
    
    # Get text with layout information
    blocks = page.get_text("dict")["blocks"]
    
    layout_info = {
        'method': 'digital_extraction',
        'block_count': len(blocks),
        'confidence': 1.0,  # Digital text is 100% confident
        'device': 'cpu'  # Digital extraction doesn't use GPU
    }
    
    return text, layout_info

def perform_ocr_on_page(page, page_num: int, ocr_engine=None) -> Tuple[str, dict]:
    """
    Perform OCR on a PDF page using EasyOCR (MPS/CUDA/CPU-accelerated)
    
    Args:
        page: PyMuPDF page object
        page_num: Page number (0-indexed)
        ocr_engine: EasyOCR Reader instance (uses global OCR_ENGINE if None)
    
    Returns:
        Tuple of (extracted_text, ocr_metadata)
    """
    if ocr_engine is None:
        ocr_engine = OCR_ENGINE
    
    # Convert PDF page to image at high DPI for better OCR
    pix = page.get_pixmap(dpi=300)
    img_data = pix.tobytes("png")
    
    # Convert to PIL Image
    from io import BytesIO
    img = Image.open(BytesIO(img_data))
    
    # Convert to numpy array for EasyOCR
    img_array = np.array(img)
    
    # Perform OCR with EasyOCR
    # Returns list of (bbox, text, confidence) tuples
    # EasyOCR automatically uses the device specified during initialization (CUDA/MPS/CPU)
    result = ocr_engine.readtext(img_array)
    
    # Extract text and confidence scores
    extracted_texts = []
    confidences = []
    
    for detection in result:
        # EasyOCR returns: (bbox, text, confidence)
        # bbox is list of 4 corner points [[x1,y1], [x2,y2], [x3,y3], [x4,y4]]
        # text is the recognized text
        # confidence is float between 0 and 1
        bbox, text, confidence = detection
        extracted_texts.append(text)
        confidences.append(confidence)
    
    # Combine all text with newlines
    full_text = '\n'.join(extracted_texts)
    
    # Calculate confidence statistics
    if confidences:
        avg_confidence = sum(confidences) / len(confidences)
        min_confidence = min(confidences)
        max_confidence = max(confidences)
    else:
        avg_confidence = 0.0
        min_confidence = 0.0
        max_confidence = 0.0
    
    # Count text regions
    text_region_count = len(extracted_texts)
    
    ocr_metadata = {
        'method': 'easyocr',
        'page_num': page_num,
        'avg_confidence': round(avg_confidence, 4),
        'min_confidence': round(min_confidence, 4),
        'max_confidence': round(max_confidence, 4),
        'text_region_count': text_region_count,
        'text_length': len(full_text),
        'image_dpi': 300,
        'device': PipelineConfig.OCR_DEVICE,
        'gpu_enabled': PipelineConfig.OCR_USE_GPU
    }
    
    return full_text, ocr_metadata

def process_document_pages(file_path: Path, version_id: str, conn) -> dict:
    """
    Process all pages in a document: extract text or perform OCR
    
    Returns:
        dict with processing summary
    """
    try:
        # Get classification data from database
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute("""
                SELECT metadata, is_scanned
                FROM document_versions
                WHERE version_id = %s
            """, (version_id,))
            result = cur.fetchone()
            
            if not result:
                raise ValueError(f"Version {version_id} not found in database")
            
            metadata = result['metadata'] or {}
            page_classifications = metadata.get('page_classifications', [])
        
        if not page_classifications:
            raise ValueError(f"No page classifications found for {version_id}")
        
        # Open document
        doc = fitz.open(file_path)
        
        all_pages_text = []
        all_pages_metadata = []
        ocr_confidence_scores = []
        
        print(f"   Processing {len(doc)} pages...")
        
        for page_num in range(len(doc)):
            page = doc[page_num]
            page_class = page_classifications[page_num]
            
            needs_ocr = page_class.get('needs_ocr', False)
            
            if needs_ocr:
                # Perform OCR using EasyOCR (GPU-accelerated on MPS/CUDA)
                text, page_metadata = perform_ocr_on_page(page, page_num)
                ocr_confidence_scores.append(page_metadata['avg_confidence'])
            else:
                # Extract digital text
                text, page_metadata = extract_text_digital(page)
            
            all_pages_text.append(text)
            all_pages_metadata.append(page_metadata)
        
        doc.close()
        
        # Combine all text
        full_text = '\n\n'.join(all_pages_text)
        
        # Calculate document-level OCR confidence
        if ocr_confidence_scores:
            doc_ocr_confidence = sum(ocr_confidence_scores) / len(ocr_confidence_scores)
        else:
            doc_ocr_confidence = 1.0  # Fully digital
        
        # Compute text hash
        text_hash = compute_text_hash(full_text)
        
        # Save extracted text to file
        text_output_path = PipelineConfig.OCR_OUTPUT_DIR / f"{version_id}_text.txt"
        with open(text_output_path, 'w', encoding='utf-8') as f:
            f.write(full_text)
        
        # Save page-level metadata
        metadata_output_path = PipelineConfig.OCR_OUTPUT_DIR / f"{version_id}_metadata.json"
        with open(metadata_output_path, 'w', encoding='utf-8') as f:
            json.dump({
                'version_id': version_id,
                'total_pages': len(doc),
                'doc_ocr_confidence': doc_ocr_confidence,
                'pages': all_pages_metadata
            }, f, indent=2)
        
        # Update database
        with conn.cursor() as cur:
            cur.execute("""
                UPDATE document_versions
                SET text_hash = %s,
                    ocr_conf_doc = %s,
                    metadata = COALESCE(metadata, '{}'::jsonb) || %s::jsonb
                WHERE version_id = %s
            """, (
                text_hash,
                doc_ocr_confidence,
                json.dumps({
                    'text_extraction': {
                        'text_output_path': str(text_output_path),
                        'metadata_output_path': str(metadata_output_path),
                        'full_text_length': len(full_text),
                        'ocr_confidence': doc_ocr_confidence,
                        'ocr_engine': PipelineConfig.OCR_ENGINE_NAME,
                        'ocr_device': PipelineConfig.OCR_DEVICE,
                        'gpu_enabled': PipelineConfig.OCR_USE_GPU
                    }
                }),
                version_id
            ))
            conn.commit()
        
        summary = {
            'version_id': version_id,
            'total_pages': len(all_pages_text),
            'ocr_pages': len(ocr_confidence_scores),
            'digital_pages': len(all_pages_text) - len(ocr_confidence_scores),
            'text_length': len(full_text),
            'ocr_confidence': round(doc_ocr_confidence, 4),
            'text_output': str(text_output_path),
            'metadata_output': str(metadata_output_path)
        }
        
        return summary
        
    except Exception as e:
        error_msg = f"Text extraction failed: {str(e)}"
        log_error(conn, version_id, "text_extraction", error_msg, {'file': str(file_path)})
        raise

print(f"✅ OCR and layout parsing functions loaded (EasyOCR with {DEVICE.upper()} acceleration)")


✅ OCR and layout parsing functions loaded (EasyOCR with MPS acceleration)


In [43]:
# Process all documents: extract text and perform OCR where needed

import time
from datetime import timedelta

# Get documents that need processing
with conn.cursor(cursor_factory=RealDictCursor) as cur:
    cur.execute("""
        SELECT version_id, file_path, is_scanned
        FROM document_versions
        WHERE status = 'draft'
        ORDER BY version_id
    """)
    documents_to_process = cur.fetchall()

if not documents_to_process:
    error_msg = "No documents found for text extraction"
    print(f"❌ {error_msg}")
    log_error(conn, None, "text_extraction", error_msg)
    raise RuntimeError(error_msg)

total_documents = len(documents_to_process)
print(f"📄 Processing {total_documents} documents for text extraction...\n")
print(f"{'='*80}")

extraction_results = []
overall_start_time = time.time()
doc_processing_times = []

for doc_idx, doc in enumerate(documents_to_process, 1):
    version_id = doc['version_id']
    file_path = Path(doc['file_path'])
    is_scanned = doc['is_scanned']
    
    try:
        # Get file from storage
        storage_path = PipelineConfig.RAW_FILES_DIR / f"{version_id}_{file_path.name}"
        if not storage_path.exists():
            storage_path = file_path
        
        if not storage_path.exists():
            raise FileNotFoundError(f"File not found: {storage_path}")
        
        print(f"\n📖 Document {doc_idx}/{total_documents}: {file_path.name}")
        print(f"   Version ID: {version_id}")
        print(f"   Needs OCR: {is_scanned}")
        print(f"   Documents remaining: {total_documents - doc_idx}")
        
        # Calculate ETA based on previous documents
        if doc_processing_times:
            avg_time_per_doc = sum(doc_processing_times) / len(doc_processing_times)
            remaining_docs = total_documents - doc_idx
            eta_seconds = avg_time_per_doc * remaining_docs
            eta_str = str(timedelta(seconds=int(eta_seconds)))
            print(f"   Estimated time remaining: {eta_str}")
        
        print(f"\n   {'─'*76}")
        
        doc_start_time = time.time()
        
        # Get classification data from database
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute("""
                SELECT metadata, is_scanned, page_count
                FROM document_versions
                WHERE version_id = %s
            """, (version_id,))
            result = cur.fetchone()
            
            if not result:
                raise ValueError(f"Version {version_id} not found in database")
            
            metadata = result['metadata'] or {}
            page_count = result['page_count']
            page_classifications = metadata.get('page_classifications', [])
        
        if not page_classifications:
            raise ValueError(f"No page classifications found for {version_id}")
        
        # Open document
        doc_pdf = fitz.open(storage_path)
        
        all_pages_text = []
        all_pages_metadata = []
        ocr_confidence_scores = []
        
        total_pages = len(doc_pdf)
        print(f"   Processing {total_pages} pages...")
        
        page_start_time = time.time()
        
        for page_num in range(total_pages):
            page = doc_pdf[page_num]
            page_class = page_classifications[page_num]
            
            needs_ocr = page_class.get('needs_ocr', False)
            
            # Show progress for current page
            progress_bar = '█' * int((page_num + 1) / total_pages * 40)
            progress_bar = progress_bar.ljust(40, '░')
            progress_pct = ((page_num + 1) / total_pages) * 100
            
            # Calculate page ETA
            if page_num > 0:
                elapsed = time.time() - page_start_time
                avg_time_per_page = elapsed / page_num
                remaining_pages = total_pages - (page_num + 1)
                page_eta_seconds = avg_time_per_page * remaining_pages
                page_eta_str = f"{int(page_eta_seconds)}s"
            else:
                page_eta_str = "calculating..."
            
            method_label = "OCR" if needs_ocr else "Digital"
            print(f"\r   [{progress_bar}] {progress_pct:5.1f}% | Page {page_num + 1}/{total_pages} ({method_label}) | ETA: {page_eta_str}  ", end='', flush=True)
            
            if needs_ocr:
                # Perform OCR using PaddleOCR
                text, page_metadata = perform_ocr_on_page(page, page_num)
                ocr_confidence_scores.append(page_metadata['avg_confidence'])
            else:
                # Extract digital text
                text, page_metadata = extract_text_digital(page)
            
            all_pages_text.append(text)
            all_pages_metadata.append(page_metadata)
        
        # Final progress line
        print(f"\r   [{'█' * 40}] 100.0% | Page {total_pages}/{total_pages} | ✅ Complete!               ")
        
        doc_pdf.close()
        
        # Combine all text
        full_text = '\n\n'.join(all_pages_text)
        
        # Calculate document-level OCR confidence (EasyOCR uses 0-1 scale)
        if ocr_confidence_scores:
            doc_ocr_confidence = float(sum(ocr_confidence_scores) / len(ocr_confidence_scores))
        else:
            doc_ocr_confidence = 1.0  # Fully digital
        
        # Compute text hash
        text_hash = compute_text_hash(full_text)
        
        # Save extracted text to file
        text_output_path = PipelineConfig.OCR_OUTPUT_DIR / f"{version_id}_text.txt"
        with open(text_output_path, 'w', encoding='utf-8') as f:
            f.write(full_text)
        
        # Save page-level metadata
        metadata_output_path = PipelineConfig.OCR_OUTPUT_DIR / f"{version_id}_metadata.json"
        with open(metadata_output_path, 'w', encoding='utf-8') as f:
            json.dump({
                'version_id': version_id,
                'total_pages': total_pages,
                'doc_ocr_confidence': doc_ocr_confidence,
                'pages': all_pages_metadata
            }, f, indent=2)
        
        # Update database
        with conn.cursor() as cur:
            cur.execute("""
                UPDATE document_versions
                SET text_hash = %s,
                    ocr_conf_doc = %s,
                    metadata = COALESCE(metadata, '{}'::jsonb) || %s::jsonb
                WHERE version_id = %s
            """, (
                text_hash,
                doc_ocr_confidence,
                json.dumps({
                    'text_extraction': {
                        'text_output_path': str(text_output_path),
                        'metadata_output_path': str(metadata_output_path),
                        'full_text_length': len(full_text),
                        'ocr_confidence': float(doc_ocr_confidence),
                        'ocr_engine': 'EasyOCR',
                        'gpu_enabled': PipelineConfig.OCR_USE_GPU
                    }
                }),
                version_id
            ))
            conn.commit()
        
        doc_elapsed_time = time.time() - doc_start_time
        doc_processing_times.append(doc_elapsed_time)
        
        extraction_results.append({
            'file': file_path.name,
            'version_id': version_id,
            'total_pages': len(all_pages_text),
            'ocr_pages': len(ocr_confidence_scores),
            'digital_pages': len(all_pages_text) - len(ocr_confidence_scores),
            'text_length': len(full_text),
            'ocr_confidence': round(doc_ocr_confidence, 2),
            'processing_time_sec': round(doc_elapsed_time, 1),
            'status': 'success'
        })
        
        # Log event
        log_event(conn, version_id, "text_extracted", {
            'total_pages': len(all_pages_text),
            'ocr_pages': len(ocr_confidence_scores),
            'text_length': len(full_text),
            'ocr_confidence': doc_ocr_confidence,
            'processing_time': doc_elapsed_time
        })
        
        print(f"\n   ✅ Extracted {len(full_text):,} characters")
        print(f"   📄 OCR pages: {len(ocr_confidence_scores)}/{len(all_pages_text)}")
        print(f"   📊 OCR confidence: {doc_ocr_confidence:.2%}")
        print(f"   ⏱️  Processing time: {doc_elapsed_time:.1f}s ({doc_elapsed_time/total_pages:.2f}s per page)")
        print(f"   {'─'*76}")
        
    except Exception as e:
        print(f"\n   ❌ Extraction failed: {e}")
        extraction_results.append({
            'file': file_path.name,
            'version_id': version_id,
            'status': 'failed',
            'error': str(e)
        })
        continue

# Overall completion
total_elapsed_time = time.time() - overall_start_time
print(f"\n{'='*80}")
print(f"⏱️  Total processing time: {str(timedelta(seconds=int(total_elapsed_time)))}")
print(f"{'='*80}")

# Create summary DataFrame
df_extraction = pd.DataFrame(extraction_results)

print(f"\n📊 Text Extraction Summary")
print(f"{'='*80}")
print(f"Total documents: {len(documents_to_process)}")
print(f"Successfully processed: {len(df_extraction[df_extraction['status'] == 'success'])}")
if 'total_pages' in df_extraction.columns:
    print(f"Total pages processed: {df_extraction['total_pages'].sum()}")
    print(f"Pages requiring OCR: {df_extraction['ocr_pages'].sum()}")
    print(f"Total characters extracted: {df_extraction['text_length'].sum():,}")
    print(f"Average OCR confidence: {df_extraction['ocr_confidence'].mean():.2%}")
    if 'processing_time_sec' in df_extraction.columns:
        print(f"Average processing time: {df_extraction['processing_time_sec'].mean():.1f}s per document")
print(f"{'='*80}\n")

# Display results
print(df_extraction.to_string(index=False))

# Save results
extraction_path = PipelineConfig.STORAGE_DIR / "extraction_results.csv"
df_extraction.to_csv(extraction_path, index=False)
print(f"\n✅ Extraction results saved to: {extraction_path}")

# Check OCR confidence threshold (PaddleOCR uses 0-1 scale)
if 'ocr_confidence' in df_extraction.columns:
    low_confidence = df_extraction[df_extraction['ocr_confidence'] < PipelineConfig.OCR_CONFIDENCE_THRESHOLD]
    if len(low_confidence) > 0:
        warning_msg = f"⚠️  {len(low_confidence)} document(s) have OCR confidence below threshold ({PipelineConfig.OCR_CONFIDENCE_THRESHOLD:.0%})"
        print(f"\n{warning_msg}")
        for idx, row in low_confidence.iterrows():
            print(f"   - {row['file']}: {row['ocr_confidence']:.2%}")
        
        log_event(conn, None, "low_ocr_confidence_warning", {
            'threshold': PipelineConfig.OCR_CONFIDENCE_THRESHOLD,
            'documents': low_confidence[['file', 'version_id', 'ocr_confidence']].to_dict('records')
        })
        
        # Note: We log the warning but don't stop the pipeline
        print(f"\n   Pipeline will continue, but these documents may need review.")

# Check for failures
if 'status' in df_extraction.columns:
    failed = df_extraction[df_extraction['status'] == 'failed']
    if len(failed) > 0:
        error_msg = f"{len(failed)} document(s) failed text extraction"
        print(f"\n❌ {error_msg}")
        log_error(conn, None, "extraction_complete", error_msg,
                  {'failed': failed['file'].tolist()})
        raise RuntimeError(error_msg)

print(f"\n✅ Layout parsing & OCR complete!")

📄 Processing 4 documents for text extraction...


📖 Document 1/4: Acts Published for the Implementation of the Constitution Corrigenda.pdf
   Version ID: v_2b308b60b512
   Needs OCR: True
   Documents remaining: 3

   ────────────────────────────────────────────────────────────────────────────
   Processing 80 pages...
   [████████████████████████████████████████] 100.0% | Page 80/80 | ✅ Complete!               

   ✅ Extracted 476,513 characters
   📄 OCR pages: 80/80
   📊 OCR confidence: 72.35%
   ⏱️  Processing time: 732.4s (9.16s per page)
   ────────────────────────────────────────────────────────────────────────────

📖 Document 2/4: Advocates Act.pdf
   Version ID: v_7f22ccad09a4
   Needs OCR: False
   Documents remaining: 2
   Estimated time remaining: 0:24:24

   ────────────────────────────────────────────────────────────────────────────
   Processing 38 pages...
   [████████████████████████████████████████] 100.0% | Page 80/80 | ✅ Complete!               ..  

   ✅ Extracted 4

---
### ✋ CHECKPOINT 5

**Layout parsing & OCR complete!**

- ✅ Text extracted from all pages
- ✅ OCR performed on scanned/hybrid pages using Tesseract
- ✅ OCR confidence scores calculated
- ✅ Extracted text saved to storage
- ✅ Text hashes computed
- ✅ Results logged to database

**Next step:** Noise removal & normalization

**⚠️ Do NOT proceed to the next cell until instructed.**

---